# 02 - Data preprocessing

Cleans and merges the collected catalogues into the training tables, derives the R_eff label, and writes the train/test split files that notebook 03 consumes. Cell order is preserved from the original `habitability_pipeline.ipynb`; outputs are cleared.

In [ ]:
# R_eff = lambda_big * p_cme * (D_REF / d)^2 ; High >= 1, Medium >= 0.1, else Low
def r_eff(lam, d, p=CME_P, dref=D_REF):
    return np.asarray(lam, float) * p * (dref / np.asarray(d, float)) ** 2

def to_label(r, thr_h=THR_HIGH, thr_m=THR_MED):
    r = np.asarray(r, float)
    return np.where(r >= thr_h, "High", np.where(r >= thr_m, "Medium", "Low"))

def mc_probs(df, n_draw=N_DRAW, seed=SEED, chunk=500):
    """Probability of Low/Medium/High per planet: draw the rate from its uncertainty and p_cme from CME_RANGE."""
    rng = np.random.default_rng(seed)
    out = np.zeros((len(df), 3))
    for i in range(0, len(df), chunk):
        d = df.iloc[i:i + chunk]; n = len(d)
        use_gamma = d["gam_shape"].notna().values[:, None]
        lam_g = rng.gamma(d["gam_shape"].fillna(1.0).values[:, None], d["gam_scale"].fillna(1.0).values[:, None], size=(n, n_draw))
        lam_n = np.clip(rng.normal(d["flare_rate_month"].values[:, None], d["lam_sigma"].fillna(0).values[:, None], size=(n, n_draw)), 0, None)
        lam = np.where(use_gamma, lam_g, lam_n)
        p = rng.uniform(CME_RANGE[0], CME_RANGE[1], size=(n, n_draw))
        r = lam * p * (D_REF / d["koi_sma"].values[:, None]) ** 2
        out[i:i + n, 0] = (r < THR_MED).mean(1)
        out[i:i + n, 2] = (r >= THR_HIGH).mean(1)
        out[i:i + n, 1] = 1 - out[i:i + n, 0] - out[i:i + n, 2]
    return out

train = allrows.copy()
train["R_eff"]          = r_eff(train["flare_rate_month"], train["koi_sma"])
train["label"]          = to_label(train["R_eff"])
train["baseline_label"] = to_label(train["flare_rate_month"])        # distance-blind Tilley-style rule
probs = mc_probs(train)
train["P_Low"], train["P_Medium"], train["P_High"] = probs[:, 0], probs[:, 1], probs[:, 2]

print(train["label"].value_counts())
print(pd.crosstab(train["rate_basis"], train["label"]))

In [ ]:
print(train[["koi_sma","flare_rate_month","flare_rate_all","max_logE","star_type","R_eff"]].describe().T)
print("rows with missing max_logE (Howard):", train["max_logE"].isna().sum())

cols = ["source","star_id","planet","koi_sma","star_type","Teff","n_all","n_big","max_logE","obs_months",
        "flare_rate_all","flare_rate_month","rate_basis","R_eff","label","baseline_label","P_Low","P_Medium","P_High"]
train[cols].to_csv(P("training_v5.csv"), index=False)
print("saved training_v5.csv:", len(train), "rows")

## Earlier label ideas, not used in the final dataset


In [ ]:
lam_, d_ = train["flare_rate_month"].values, train["koi_sma"].values
ref_lab = to_label(r_eff(lam_, d_))
variants = {
    "D_REF 0.10 AU":       dict(dref=0.10),
    "D_REF 0.25 AU":       dict(dref=0.25),
    "p_cme 0.35":          dict(p=0.35),
    "p_cme 0.60":          dict(p=0.60),
    "thresholds x0.5":     dict(thr_h=0.5, thr_m=0.05),
    "thresholds x2":       dict(thr_h=2.0, thr_m=0.2),
}
rows = []
for name, kw in variants.items():
    r = r_eff(lam_, d_, p=kw.get("p", CME_P), dref=kw.get("dref", D_REF))
    lab = to_label(r, kw.get("thr_h", THR_HIGH), kw.get("thr_m", THR_MED))
    rows.append({"variant": name, "% labels changed": 100 * (lab != ref_lab).mean(),
                 "High": (lab == "High").sum(), "Medium": (lab == "Medium").sum(), "Low": (lab == "Low").sum()})
print("reference counts:", pd.Series(ref_lab).value_counts().to_dict())
print(pd.DataFrame(rows).round(1))

In [ ]:
# Inputs come from the papers quoted in Tilley et al. 2019 (VERIFY against Vida et al. 2017 and Davenport et al. 2016):
#   TRAPPIST-1: >= 0.75 flares/day above 1.26e30 erg.   Proxima Cen: about one 10^31.5 erg flare per week.
# The flare-frequency slope (alpha) is not measured here, so a range is used. Planet distances: VERIFY.
cases = {
    "TRAPPIST-1 e": dict(d=0.0292, e_ref=1.26e30,   nu_ref_per_day=0.75),
    "TRAPPIST-1 f": dict(d=0.0385, e_ref=1.26e30,   nu_ref_per_day=0.75),
    "TRAPPIST-1 g": dict(d=0.0469, e_ref=1.26e30,   nu_ref_per_day=0.75),
    "Proxima Cen b": dict(d=0.0485, e_ref=10**31.5, nu_ref_per_day=1/7),
}
rows = []
for name, c in cases.items():
    for alpha in (1.6, 2.0, 2.4):
        lam_big = c["nu_ref_per_day"] * (1e34 / c["e_ref"]) ** (1 - alpha) * DAYS_PER_MONTH   # flares >= 1e34 per month
        r = r_eff(lam_big, c["d"])
        rows.append({"planet": name, "alpha": alpha, "lambda_big_per_month": lam_big, "R_eff": float(r),
                     "label": str(to_label(r)), "Tilley-style (distance-blind)": str(to_label(lam_big))})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
T_ = [3000,3400,3700,4000,4500,5000,5500,5800,6200,6600,7200]          # approximate main-sequence Teff -> luminosity
L_ = [0.0015,0.012,0.03,0.08,0.2,0.4,0.75,1.0,1.8,3.0,6.0]             # (state in report); HZ distance = sqrt(L)
DIST_GRID = [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]                            # AU

st = pd.concat(star_tabs.values(), ignore_index=True)
st["star_id"] = st["star_key"]
st["Teff"] = resolve_teff(st)
st = st.dropna(subset=["Teff"])
st["prio"] = st["source"].map({s: i for i, s in enumerate(PRIORITY)})
st = st[st["prio"] == st.groupby("star_id")["prio"].transform("min")].drop_duplicates("star_id")
st = st[~st["star_id"].isin(train["star_id"])]                          # keep real-planet stars out of training

parts = []
for dv in DIST_GRID:
    t = st.copy(); t["koi_sma"] = dv; t["dist_kind"] = "grid"; parts.append(t)
t = st.copy(); t["koi_sma"] = np.sqrt(np.interp(t["Teff"], T_, L_)); t["dist_kind"] = "HZ"; parts.append(t)
scen = pd.concat(parts, ignore_index=True)

scen["star_type"]      = np.select([scen["Teff"] > 6000, scen["Teff"] > 5200, scen["Teff"] > 3700], [0, 1, 2], default=3)
scen["R_eff"]          = r_eff(scen["flare_rate_month"], scen["koi_sma"])
scen["label"]          = to_label(scen["R_eff"])
scen["baseline_label"] = to_label(scen["flare_rate_month"])
print(len(scen), "scenario rows from", scen["star_id"].nunique(), "stars")
print(scen["label"].value_counts())

In [ ]:
Xs, ys = scen[FEATS].astype(float).values, scen["label"].map(LAB).values
Xv, yv = train[FEATS].astype(float).values, train["label"].map(LAB).values
bv = train["baseline_label"].map(LAB).values
print("Tilley-style rule on real planets: macroF1 =", round(f1_score(yv, bv, average="macro", labels=[0,1,2], zero_division=0), 3))
pred_v = {}
for name in ["RandomForest", "XGBoost", "MLP"]:
    pred_v[name] = fit_predict(name, Xs, ys, Xv)
    print(f"{name:13s} macroF1 on real planets = {f1_score(yv, pred_v[name], average='macro', labels=[0,1,2], zero_division=0):.3f}")
best_v = max(pred_v, key=lambda k: f1_score(yv, pred_v[k], average="macro", labels=[0,1,2], zero_division=0))
print(classification_report(yv, pred_v[best_v], labels=[0,1,2], target_names=NAMES, zero_division=0))
print(pd.DataFrame(confusion_matrix(yv, pred_v[best_v], labels=[0,1,2]),
                   index=[f"true {n}" for n in NAMES], columns=[f"pred {n}" for n in NAMES]))

In [ ]:
import os, glob
import numpy as np, pandas as pd
from io import StringIO
assert "mp" in globals(), "run the master planet table cell first"

FOLDER = r"D:\Research\data"
def P(n): return os.path.join(FOLDER, n)
D_REF, CME_P, THR_HIGH, THR_MED = 0.16, 0.47, 1.0, 0.1          # fixed label settings (same as v5)
LIN_E = "Ef"                                                      # fixed choice: observed Lin energy (Ef-cal differs by about 1%)
PRIORITY = ["Kepler", "Lin2024", "A103", "Howard2022", "Stelzer2022"]   # one source per star; decided before seeing results

def r_eff(lam, d, p=CME_P, dref=D_REF): return np.asarray(lam, float) * p * (dref / np.asarray(d, float)) ** 2
def to_label(r, th=THR_HIGH, tm=THR_MED):
    r = np.asarray(r, float); return np.where(r >= th, "High", np.where(r >= tm, "Medium", "Low"))

obs = pd.read_csv(P("mast_obs_times.csv"))

# ---------- Lin 2024: flares per star (energy Ef in erg) and star temperature (table 10) ----------
rows = [l for l in open(P("ajad7956t9_mrt.txt"), errors="ignore") if l[:10].strip().isdigit()]
lin = pd.read_fwf(StringIO("".join(rows)), header=None, colspecs=[(0, 10), (46, 55), (56, 65)], names=["TIC", "Ef", "Ef_cal"])
lin = lin.apply(pd.to_numeric, errors="coerce").dropna(subset=["TIC"]); lin["TIC"] = lin["TIC"].astype("int64")
lin["E"] = lin["Ef" if LIN_E == "Ef" else "Ef_cal"]
g_lin = lin.groupby("TIC").agg(n_all=("E", "size"), n_big=("E", lambda s: int((s >= 1e34).sum())), max_E=("E", "max"))
rows10 = [l for l in open(P("ajad7956t10_mrt.txt"), errors="ignore") if l[:10].strip().isdigit()]
t10 = pd.read_fwf(StringIO("".join(rows10)), header=None, colspecs=[(0, 10), (56, 63)], names=["TIC", "Teff"]).apply(pd.to_numeric, errors="coerce")
g_lin = g_lin.join(t10.dropna(subset=["TIC"]).astype({"TIC": "int64"}).drop_duplicates("TIC").set_index("TIC"))

# ---------- A&A 689 A103: flares per star (Ebol in erg; "-" = no energy) ----------
a = pd.read_csv(sorted(glob.glob(P("a103_*.csv")))[0])
a["TIC"] = pd.to_numeric(a["TIC"], errors="coerce"); a = a.dropna(subset=["TIC"]); a["TIC"] = a["TIC"].astype("int64")
a["E"] = pd.to_numeric(a["Ebol"], errors="coerce"); a["Teff"] = pd.to_numeric(a["Teff"], errors="coerce")
g_a = a.groupby("TIC").agg(n_all=("E", "size"), n_big=("E", lambda s: int((s >= 1e34).sum())), max_E=("E", "max"), Teff=("Teff", "first"))

def star_rows(g, src, mcol):
    d = g.reset_index().merge(obs[["TIC", mcol]].rename(columns={mcol: "obs_months"}), on="TIC", how="left")
    bad = int((d["obs_months"].isna() | (d["obs_months"] <= 0)).sum())
    print(f"{src:8s} stars: {len(d):6d} | without an observing time (dropped): {bad}")
    d["source"] = src
    return d[d["obs_months"] > 0]

ns = pd.concat([star_rows(g_lin, "Lin2024", "lin_months"), star_rows(g_a, "A103", "a103_months")], ignore_index=True)
ns["flare_rate_all"]   = ns["n_all"] / ns["obs_months"]
ns["flare_rate_month"] = ns["n_big"] / ns["obs_months"]                 # observed flares >= 1e34 per month (0 if none seen)
ns["max_logE"]   = np.log10(ns["max_E"])
ns["rate_basis"] = np.where(ns["n_big"] > 0, "count", "count_zero")
ns["star_id"]    = "TIC" + ns["TIC"].astype(str)
ns["obs_basis"]  = "MAST sectors"
new_p = ns.merge(mp[["tic", "planet", "koi_sma", "dist_basis"]], left_on="TIC", right_on="tic", how="inner")
print("new planet rows (Lin + A103 stars matched to the master planet table):", len(new_p))

# ---------- older sources kept from v5: Kepler, Howard, Stelzer (Gunther and Yang 2023 are replaced by Lin and A103) ----------
old = pd.read_csv(P("training_v5.csv"))
old = old[old["source"].isin(["Kepler", "Howard2022", "Stelzer2022"])].copy()
z = old["rate_basis"] == "extrap"                                          # no >= 1e34 flare seen: use the observed count (0), not a fitted extrapolation
old.loc[z, "flare_rate_month"] = 0.0; old.loc[z, "rate_basis"] = "count_zero"
old["dist_basis"] = "catalogued"
old["obs_basis"] = old["source"].map({"Kepler": "assumed 51 months", "Howard2022": "catalogue", "Stelzer2022": "catalogue"})

COLS = ["source", "star_id", "planet", "koi_sma", "Teff", "n_all", "n_big", "max_logE", "obs_months",
        "flare_rate_all", "flare_rate_month", "rate_basis", "dist_basis", "obs_basis"]
allr = pd.concat([old[COLS], new_p[COLS]], ignore_index=True)
allr["prio"] = allr["source"].map({s: i for i, s in enumerate(PRIORITY)})
allr = allr[allr["prio"] == allr.groupby("star_id")["prio"].transform("min")].drop_duplicates(["star_id", "planet"]).drop(columns="prio")
allr = allr.dropna(subset=["Teff", "koi_sma", "flare_rate_month"]); allr = allr[allr["koi_sma"] > 0].copy()

allr["star_type"] = np.select([allr["Teff"] > 6000, allr["Teff"] > 5200, allr["Teff"] > 3700], [0, 1, 2], default=3)
allr["R_eff"] = r_eff(allr["flare_rate_month"], allr["koi_sma"])
allr["label"] = to_label(allr["R_eff"])
allr["baseline_label"] = to_label(allr["flare_rate_month"])                # distance-blind Tilley-style rule
allr.reset_index(drop=True).to_csv(P("training_v6.csv"), index=False)

print("\nTOTAL real planet rows:", len(allr), "| unique stars:", allr["star_id"].nunique())
print(allr["source"].value_counts().to_string()); print(allr["label"].value_counts().to_string())
print(pd.crosstab(allr["rate_basis"], allr["label"]))
print(allr["dist_basis"].value_counts().to_string()); print(allr["obs_basis"].value_counts().to_string())
print("\nkoi_sma percentiles:", allr["koi_sma"].quantile([0, .01, .5, .99, 1]).round(4).tolist())
print(allr.nlargest(5, "koi_sma")[["source", "planet", "koi_sma", "dist_basis"]].to_string(index=False))
print(allr.nlargest(5, "flare_rate_month")[["source", "planet", "n_big", "obs_months", "flare_rate_month"]].to_string(index=False))

In [ ]:
v5 = pd.read_csv(P("training_v5.csv")); v6 = pd.read_csv(P("training_v6.csv"))
print("v5 rows:", len(v5), "| v6 rows:", len(v6))
lost = v5[~v5["star_id"].isin(v6["star_id"])]
print("v5 rows whose star is not in v6:", len(lost), "| stars:", lost["star_id"].nunique())
print(lost["source"].value_counts().to_string())
print("v5 rows whose star is also in v6:", int(v5["star_id"].isin(v6["star_id"]).sum()))

In [ ]:
lost_tic = {int(s[3:]) for s in lost["star_id"] if s.startswith("TIC")}
ok_obs = set(obs.loc[obs["lin_months"] > 0, "TIC"])
print("lost stars:", len(lost_tic), "| in Lin:", len(lost_tic & set(g_lin.index)), "| in A103:", len(lost_tic & set(g_a.index)),
      "| in master planet table:", len(lost_tic & set(mp["tic"])), "| with a MAST time:", len(lost_tic & ok_obs))
print(lost[lost["source"] == "Howard2022"][["star_id", "planet", "Teff", "koi_sma", "flare_rate_month", "rate_basis"]])

In [ ]:
import os
import pandas as pd
FOLDER = r"D:\Research\data"
v5 = pd.read_csv(os.path.join(FOLDER, "training_v5.csv"))

print("rows:", len(v5), "| unique stars:", v5["star_id"].nunique(), "| columns:", v5.shape[1])
print(v5.columns.tolist())
print("\nby source:\n", v5["source"].value_counts().to_string())
print("\nby label:\n", v5["label"].value_counts().to_string())
print("\nsource x label:\n", pd.crosstab(v5["source"], v5["label"]))
print("\nrate basis x label:\n", pd.crosstab(v5["rate_basis"], v5["label"]))
print("\nmissing values:\n", v5.isna().sum()[v5.isna().sum() > 0].to_string())
print("\nkoi_sma percentiles:", v5["koi_sma"].quantile([0, .01, .5, .99, 1]).round(4).tolist())
print("flare_rate_month percentiles:", v5["flare_rate_month"].quantile([0, .5, .99, 1]).round(3).tolist())
print("duplicate star_id + planet rows:", int(v5.duplicated(["star_id", "planet"]).sum()))
print(v5.head())

In [ ]:
import pandas as pd
v5  = pd.read_csv(r"D:\Research\data\training_v5.csv")
new = pd.read_csv(r"D:\Research\data\training_v6.csv")

# check the columns match before merging
print("in v5 but not in new:", set(v5.columns) - set(new.columns))
print("in new but not in v5:", set(new.columns) - set(v5.columns))

cols = list(v5.columns)
for c in cols:
    if c not in new.columns: new[c] = pd.NA
new = new[cols + [c for c in new.columns if c not in cols]]

allr = pd.concat([new, v5], ignore_index=True)
prio = {"Kepler":0,"Howard2022":1,"Lin2024":2,"A103":3,
        "Stelzer2022":4,"Gunther2020":5,"Yang2023":6}
allr["_p"] = allr["source"].map(prio)
allr["star_id"] = allr["star_id"].astype(str).str.strip()
allr = (allr.sort_values("_p")
            .drop_duplicates(["star_id","planet"], keep="first")
            .drop(columns="_p").reset_index(drop=True))

allr["obs_assumed"] = allr["source"].isin(["Kepler","Gunther2020","Yang2023"])
allr["detected_big"] = allr["n_big"] > 0

print("rows:", len(allr), "| stars:", allr.star_id.nunique())
print(allr.groupby(["source","label"]).size().unstack(fill_value=0))
print("assumed obs time:", allr.obs_assumed.sum(),
      "| rows with a >=1e34 flare:", allr.detected_big.sum())
allr.to_csv(r"D:\Research\data\training_v7.csv", index=False)

In [ ]:
new = pd.read_csv(r"D:\Research\data\training_v6.csv")
print(new.shape)

In [ ]:
d = pd.read_csv(r"D:\Research\data\training_v7.csv")
print(d["koi_sma"].describe())
print(d.nlargest(8, "koi_sma")[["source","star_id","planet","koi_sma","dist_basis"]])
print(d.nsmallest(5, "koi_sma")[["source","star_id","planet","koi_sma","dist_basis"]])
print(d["flare_rate_month"].describe())
print(d.nlargest(8, "flare_rate_month")[["source","star_id","planet","n_all","n_big","obs_months","flare_rate_month","rate_basis"]])
print("missing:", d[["koi_sma","flare_rate_month","flare_rate_all","max_logE","obs_months"]].isna().sum().to_dict())
print(d["obs_months"].describe())

In [ ]:
import pandas as pd, os
D = r"D:\Research\data"
v7 = pd.read_csv(os.path.join(D, "training_v7.csv"))
have = set(zip(v7.star_id.astype(str).str.strip(), v7.planet.astype(str).str.strip()))
files = ["training_data.csv","training_data_v2.csv","training_data_v3.csv",
         "training_data_v4.csv","training_data_v4b.csv",
         "training_real_distance.csv","training_star_level.csv",
         "validation_real_planets.csv","kepler_labeled_v2.csv",
         "tess_gunther_labeled.csv","yang_labeled.csv"]
for f in files:
    try:
        x = pd.read_csv(os.path.join(D, f))
    except Exception as e:
        print(f, "ERROR", e); continue
    print("\n==", f, x.shape)
    print(list(x.columns)[:25])
    if {"star_id","planet"} <= set(x.columns):
        pairs = set(zip(x.star_id.astype(str).str.strip(), x.planet.astype(str).str.strip()))
        print("unique planets:", len(pairs), "| NOT in v7:", len(pairs - have))
    else:
        print("no star_id+planet columns")

In [ ]:
import pandas as pd, os
D = r"D:\Research\data"
v7 = pd.read_csv(os.path.join(D, "training_v7.csv"))
have = set(zip(v7.star_id.astype(str).str.strip(), v7.planet.astype(str).str.strip()))

x = pd.read_csv(os.path.join(D, "training_real_distance.csv"))
x["star_id"] = x.star_id.astype(str).str.strip()
x["kepoi_name"] = x.kepoi_name.astype(str).str.strip()
pairs = set(zip(x.star_id, x.kepoi_name))

print({
 "rows": len(x),
 "unique_planets": len(pairs),
 "unique_stars": x.star_id.nunique(),
 "not_in_v7": len(pairs - have),
 "upper_limit": x.is_upper_limit.value_counts(dropna=False).to_dict(),
 "sources": x.source.value_counts().to_dict(),
 "sma_min_max": (x.koi_sma.min(), x.koi_sma.max()),
 "n_big_missing": int(x.n_big.isna().sum()),
})

In [ ]:
nd = x[x.is_upper_limit == 1].copy()
print({
 "rows": len(nd),
 "rate_describe": nd.flare_rate_month.describe().round(4).to_dict(),
 "rate_zero": int((nd.flare_rate_month == 0).sum()),
 "n_big_values": nd.n_big.value_counts(dropna=False).head(5).to_dict(),
 "sma_gt_10": int((nd.koi_sma > 10).sum()),
 "stars_already_in_v7": int(nd.star_id.isin(v7.star_id).sum()),
 "old_label": nd.label.value_counts().to_dict(),
})

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
v7 = pd.read_csv(rf"{D}\training_v7.csv")
v7 = v7[v7.koi_sma <= 10].copy()
v7["howard_fix"] = v7["dist_basis"].isna() & (v7["source"] == "Howard2022")
v7.loc[v7.howard_fix, "dist_basis"] = "howard_catalogued"
v7 = v7.drop(columns="howard_fix")

x = pd.read_csv(rf"{D}\training_real_distance.csv")
nd = x[x.is_upper_limit == 1].copy()
nd = nd[nd.koi_sma <= 10]
nd["star_id"] = nd.star_id.astype(str).str.strip()
nd["planet"] = nd.kepoi_name.astype(str).str.strip()
nd["flare_rate_month"] = 0.0
nd["flare_rate_all"] = 0.0
nd["n_big"] = 0
nd["rate_basis"] = "no_detection"
nd["obs_assumed"] = nd.source.str.startswith("Kepler")
nd["detected_big"] = False
nd["R_eff"] = 0.0
nd["label"] = "Low"
nd["baseline_label"] = "Low"
keep = [c for c in v7.columns if c in nd.columns]
nd = nd[keep]

v7["no_detection"] = False
nd["no_detection"] = True
all8 = pd.concat([v7, nd], ignore_index=True)
all8 = all8.drop_duplicates(["star_id", "planet"]).reset_index(drop=True)

print("rows:", len(all8), "| stars:", all8.star_id.nunique())
print(all8.groupby(["no_detection", "label"]).size().unstack(fill_value=0))
print("assumed obs time:", int(all8.obs_assumed.sum()))
all8.to_csv(rf"{D}\training_v8.csv", index=False)

In [ ]:
d = all8[~all8.no_detection]
print(d.flare_rate_month.describe().to_string())
print(d.nlargest(8,"flare_rate_month")[["source","planet","n_all","n_big","obs_months","flare_rate_month","rate_basis"]].to_string())
print(d[["koi_sma","flare_rate_month","flare_rate_all","max_logE","obs_months"]].isna().sum().to_string())

In [ ]:
d = all8[~all8.no_detection]
cnt = d[d.rate_basis == "count"].flare_rate_month
print("count-based max:", cnt.max(), "| 99th pct:", cnt.quantile(0.99))
how = d[d.rate_basis == "howard_ffd"].flare_rate_month
cap = cnt.max()
print("howard rows above cap:", int((how > cap).sum()), "of", len(how))
print(how.describe().round(3).to_string())


In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v8.csv")

cap = 11.450303139606332          # max of count-based rates, from your data
m = (a.rate_basis == "howard_ffd") & (a.flare_rate_month > cap)
a["rate_capped"] = m
a["flare_rate_month_raw"] = a["flare_rate_month"]
a.loc[m, "flare_rate_month"] = cap
print("capped rows:", int(m.sum()))
print(a.loc[m, ["planet","flare_rate_month_raw","flare_rate_month"]].to_string())

# recompute the label with the fixed v5 formula
P_CME = 0.47
a["R_eff"] = a["flare_rate_month"] * P_CME * (0.16 / a["koi_sma"])**2
a["label_old"] = a["label"]
a["label"] = np.where(a.R_eff >= 1, "High", np.where(a.R_eff >= 0.1, "Medium", "Low"))
a["baseline_label"] = np.where(a.flare_rate_month >= 1, "High",
                       np.where(a.flare_rate_month >= 0.1, "Medium", "Low"))

print("labels changed:", int((a.label != a.label_old).sum()))
print(pd.crosstab(a.no_detection, a.label))
a.to_csv(rf"{D}\training_v9.csv", index=False)

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedGroupKFold
D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v9.csv")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, te_idx = next(sgkf.split(a, a["label"], groups=a["star_id"]))
a["split"] = "train"
a.loc[a.index[te_idx], "split"] = "test"

print("star overlap train/test:",
      len(set(a[a.split=="train"].star_id) & set(a[a.split=="test"].star_id)))
print(pd.crosstab([a.split, a.no_detection], a.label))
print("missing in features:",
      a[["koi_sma","flare_rate_month","flare_rate_all","max_logE","star_type"]].isna().sum().to_dict())
a.to_csv(rf"{D}\training_v9_split.csv", index=False)

In [ ]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
D = r"D:\Research\data"

# 1. Pietras flares -> per-star counts
rows = [l.rstrip("\n") for l in open(rf"{D}\datafile.txt", errors="replace")
        if l[:10].strip().isdigit() and len(l) > 55]
p = pd.DataFrame({"tic": [int(l[0:10]) for l in rows],
                  "E1": [float(l[34:42]) if l[34:42].strip() else np.nan for l in rows],
                  "pdf": [l[55:56] for l in rows]})
p = p[p.pdf != "Y"]
s = p.groupby("tic").agg(n_all=("tic", "size"),
                         n_big=("E1", lambda x: int((x >= 1e34).sum())),
                         maxE=("E1", "max")).reset_index()
hosts = pd.read_csv(rf"{D}\pietras_new_hosts.csv")
m = pd.read_csv(rf"{D}\mast_obs_times_pietras.csv")
m = m[m.n_sectors.fillna(0) > 0]
s = s[s.tic.isin(hosts.tic)].merge(m[["tic", "n_sectors", "obs_months"]], on="tic")
s["flare_rate_month"] = s.n_big / s.obs_months
s["flare_rate_all"] = s.n_all / s.obs_months
s["max_logE"] = np.log10(s.maxE)
print("stars with flares + time:", len(s))

# 2. planets from the raw tables
def tic_int(x): return pd.to_numeric(x.astype(str).str.extract(r"(\d+)")[0], errors="coerce")
def col(df, c): return df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
ps = pd.read_csv(rf"{D}\PSCompPars_2026.10.16_13.03.27.csv", comment="#", low_memory=False)
k2 = pd.read_csv(rf"{D}\k2pandc_2026.10.16_13.09.00.csv", comment="#", low_memory=False)
k2 = k2[k2.default_flag == 1] if "default_flag" in k2.columns else k2
toi = pd.read_csv(rf"{D}\TOI_2026.10.16_13.07.57.csv", comment="#", low_memory=False)

def std(df, tic, name, a, per, mass, teff, prio):
    return pd.DataFrame({"tic": tic_int(tic), "planet": name, "a_cat": a, "per": per,
                         "mass": mass, "Teff": teff, "prio": prio})
pl = pd.concat([
    std(ps, ps.tic_id, col(ps, "pl_name"), col(ps, "pl_orbsmax"), col(ps, "pl_orbper"),
        col(ps, "st_mass"), col(ps, "st_teff"), 0),
    std(k2, k2.tic_id, col(k2, "pl_name"), col(k2, "pl_orbsmax"), col(k2, "pl_orbper"),
        col(k2, "st_mass"), col(k2, "st_teff"), 1),
    std(toi, toi.tid, "TOI-" + pd.to_numeric(col(toi, "toi"), errors="coerce").map("{:.2f}".format),
        np.nan, col(toi, "pl_orbper"),
        10 ** (col(toi, "st_logg") - 4.438) * col(toi, "st_rad") ** 2, col(toi, "st_teff"), 2),
], ignore_index=True)
pl = pl[pl.tic.isin(s.tic)].sort_values(["tic", "prio"])
n0 = len(pl)

keep = []
for _, g in pl.groupby("tic"):
    kept = []
    for i, r in g.iterrows():
        dup = pd.notna(r.per) and any(pd.notna(k) and k > 0 and abs(r.per - k) / k < 0.02 for k in kept)
        if not dup:
            keep.append(i)
        if pd.notna(r.per):
            kept.append(r.per)
pl = pl.loc[keep]
print("planet rows before/after dedupe:", n0, len(pl))

# 3. distance
a_calc = (pl.mass * (pl.per / 365.25) ** 2) ** (1 / 3)
pl["koi_sma"] = pl.a_cat.fillna(a_calc)
pl["dist_basis"] = np.where(pl.a_cat.notna(), "catalogued",
                   np.where(pl.prio == 2, "kepler3_logg_radius", "kepler3_period_mass"))
print("no distance:", int(pl.koi_sma.isna().sum()), "| > 10 AU:", int((pl.koi_sma > 10).sum()))
pl = pl[pl.koi_sma.notna() & (pl.koi_sma <= 10)]

# 4. join, star_type, label
d = pl.merge(s, on="tic")
d["star_id"] = "TIC" + d.tic.astype(int).astype(str)
v9 = pd.read_csv(rf"{D}\training_v9_split.csv")
print("star_type values in v9:", v9.star_type.value_counts(dropna=False).to_dict())
ref = v9[v9.Teff.notna() & v9.star_type.notna()]
knn = KNeighborsClassifier(15).fit(ref[["Teff"]], ref.star_type.astype(str))
ok = d.Teff.notna()
d.loc[ok, "star_type"] = knn.predict(d.loc[ok, ["Teff"]])
print("no Teff -> dropped:", int((~ok).sum()))
d = d[ok].copy()

d["R_eff"] = d.flare_rate_month * 0.47 * (0.16 / d.koi_sma) ** 2
d["label"] = np.where(d.R_eff >= 1, "High", np.where(d.R_eff >= 0.1, "Medium", "Low"))
d["baseline_label"] = np.where(d.flare_rate_month >= 1, "High",
                       np.where(d.flare_rate_month >= 0.1, "Medium", "Low"))
d["source"] = "Pietras2022"; d["rate_basis"] = "count"; d["obs_basis"] = "MAST sectors"
d["obs_assumed"] = False; d["no_detection"] = False; d["detected_big"] = d.n_big > 0
d["rate_capped"] = False; d["flare_rate_month_raw"] = d.flare_rate_month
d["obs_months"] = d.obs_months

print("NEW rows:", len(d), "| stars:", d.star_id.nunique())
print(d.label.value_counts().to_dict(), "| dist_basis:", d.dist_basis.value_counts().to_dict())
print("star overlap with v9:", len(set(d.star_id) & set(v9.star_id)))
print(d.nlargest(6, "flare_rate_month")[["planet", "n_all", "n_big", "obs_months",
                                         "flare_rate_month", "koi_sma", "label"]].to_string())

# 5. combine (no split yet)
d = d.drop(columns=["tic", "a_cat", "per", "mass", "prio", "maxE", "n_sectors"])
all10 = pd.concat([v9.drop(columns="split"), d], ignore_index=True)
all10 = all10.drop_duplicates(["star_id", "planet"]).reset_index(drop=True)
print("TOTAL rows:", len(all10), "| stars:", all10.star_id.nunique())
print(pd.crosstab(all10.no_detection, all10.label))
all10.to_csv(rf"{D}\training_v10.csv", index=False)

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v10.csv")
print("star_type dtype:", a.star_type.dtype, a.star_type.value_counts().to_dict())
print(a.groupby(["source" if False else a.source.eq("Pietras2022").rename("pietras"), "star_type"]).Teff
        .agg(["min", "max", "count"]).round(0).to_string())

rows = [l.rstrip("\n") for l in open(rf"{D}\datafile.txt", errors="replace")
        if l[:10].strip().isdigit() and len(l) > 55]
p = pd.DataFrame({"tic": [int(l[0:10]) for l in rows],
                  "E2": [float(l[43:51]) if l[43:51].strip() else np.nan for l in rows],
                  "pdf": [l[55:56] for l in rows]})
p = p[p.pdf != "Y"]
b2 = p.groupby("tic").agg(n_E2=("E2", lambda x: int(x.notna().sum())),
                          n_big_E2=("E2", lambda x: int((x >= 1e34).sum()))).reset_index()

n = a[a.source == "Pietras2022"].copy()
n["tic"] = n.star_id.str.replace("TIC", "").astype(int)
n = n.merge(b2, on="tic", how="left")
print("rows with no E2 energy (excluded from comparison):", int((n.n_E2.fillna(0) == 0).sum()))
n = n[n.n_E2 > 0]
n["R2"] = n.n_big_E2 / n.obs_months * 0.47 * (0.16 / n.koi_sma) ** 2
n["label_E2"] = np.where(n.R2 >= 1, "High", np.where(n.R2 >= 0.1, "Medium", "Low"))
print(pd.crosstab(n.label, n.label_E2))
print("labels changed:", int((n.label != n.label_E2).sum()), "of", len(n))
print(pd.crosstab(pd.cut(n.Teff, [0, 3800, 5000, 6000, 10000]), n.label))

In [ ]:
   from sklearn.model_selection import StratifiedGroupKFold
   a = pd.read_csv(rf"{D}\training_v10.csv")
   sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
   tr_idx, te_idx = next(sgkf.split(a, a["label"], groups=a["star_id"]))
   a["split"] = "train"
   a.loc[a.index[te_idx], "split"] = "test"
   print("star overlap:", len(set(a[a.split == "train"].star_id) & set(a[a.split == "test"].star_id)))
   print(pd.crosstab([a.split, a.no_detection], a.label))
   print(pd.crosstab(a.split, a.source))
   print("missing:", a[["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type"]].isna().sum().to_dict())
   a.to_csv(rf"{D}\training_v10_split.csv", index=False)

In [ ]:
import pandas as pd
D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v10_split.csv")
d = a[~a.no_detection].reset_index(drop=True)
print("rows:", len(d), "| stars:", d.star_id.nunique())
print(pd.crosstab(d.split, d.label))
print("star overlap:", len(set(d[d.split=="train"].star_id) & set(d[d.split=="test"].star_id)))
d.to_csv(rf"{D}\training_v10_detected.csv", index=False)

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v11.csv")
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, te_idx = next(sgkf.split(a, a["label"], groups=a["star_id"]))
a["split"] = "train"
a.loc[a.index[te_idx], "split"] = "test"
print("rows:", len(a), "| stars:", a.star_id.nunique())
print("star overlap:", len(set(a[a.split=="train"].star_id) & set(a[a.split=="test"].star_id)))
print(pd.crosstab(a.split, a.label))
print(pd.crosstab(a.split, a.source))
print("missing:", a[["koi_sma","flare_rate_month","flare_rate_all","max_logE","star_type"]].isna().sum().to_dict())
a.to_csv(rf"{D}\training_v11_split.csv", index=False)

In [ ]:
import pandas as pd, os
B = r"D:\Research\data\shi2025\stellar\stellar"
cat = pd.read_csv(os.path.join(B, "catalog.csv"))
big = cat[cat.Energy >= 1e34]
print("whole catalogue, >=1e34:", len(big), "flares, share", round(big.CME_Association.mean(), 3))

v = pd.read_csv(r"D:\Research\data\training_v11_split.csv")
def lab(p):
    R = v.flare_rate_month * p * (0.16 / v.koi_sma) ** 2
    return pd.cut(R, [-1, 0.1, 1, 1e9], labels=["Low", "Medium", "High"], right=False)
base = lab(0.47)
for p in [0.35, 0.25, 0.60]:
    ch = (lab(p) != base)
    print(f"p={p}: {int(ch.sum())} of {len(v)} labels change")

In [ ]:
import pandas as pd, numpy as np, os
from sklearn.linear_model import LogisticRegression
B = r"D:\Research\data\shi2025\stellar\stellar"
c = pd.read_csv(os.path.join(B, "catalog.csv"))
c = c[c.Energy > 0].copy()
c["logE"] = np.log10(c.Energy)
c["bin"] = pd.cut(c.logE, [0, 33, 33.5, 34, 34.5, 35, 35.5, 99])
print(c.groupby("bin", observed=True).CME_Association.agg(["size", "mean"]).round(3))
m = LogisticRegression(C=1e6).fit(c[["logE"]], c.CME_Association)
print("slope", round(m.coef_[0][0], 3), "intercept", round(m.intercept_[0], 3))
for e in [33, 34, 35, 36]:
    print(e, round(m.predict_proba(pd.DataFrame({"logE": [e]}))[0, 1], 3))

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
df = pd.read_csv(rf"{D}\training_v12.csv")

st = np.select([df.Teff > 6000, df.Teff > 5200, df.Teff > 3700], [0, 1, 2], 3)
has = df.Teff.notna()
print("Teff missing:", int((~has).sum()), "| star_type changed:", int((has & (st != df.star_type)).sum()))
print(df[has & (st != df.star_type)].groupby("source").size())
df["star_type"] = np.where(has, st, df.star_type)   # keep old value only if Teff is missing

keep = ["star_id", "planet", "source", "split", "label",
        "koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type", "Teff", "cme_prob"]
out = df[keep]
out.to_csv(rf"{D}\training_v12_model.csv", index=False)
print(out.shape, out.label.value_counts().to_dict(), out.split.value_counts().to_dict())